# DataMate Multi-Stage Code-Generation Agent (Qwen2.5-Coder-7B-Instruct) — Google Colab Pro

Notebook này được cấu hình chuyên biệt cho **Google Colab Pro (GPU A100 / L4 / T4)** để chạy toàn bộ pipeline **DataMate (Zalo AI Challenge 2026)**.

Hỗ trợ **2 chế độ vận hành**:
1. **Chế độ 1 (Chạy End-to-End 100% trên Colab Pro + Google Drive)**: Mount Google Drive cá nhân, đồng bộ thư mục `zalo AI`, kiểm thử `pytest`, nạp `Qwen/Qwen2.5-Coder-7B-Instruct` lên GPU và chạy `run_pipeline.py` + `evaluate.py`.
2. **Chế độ 2 (Colab Pro làm Remote GPU Server cho máy Mac)**: Mở cổng API tương thích OpenAI qua Cloudflare Tunnel để máy Mac tại `/Applications/zalo AI` gọi trực tiếp GPU Colab Pro.

## Bước 1: Kiểm tra GPU trên Google Colab Pro và Cài đặt thư viện

In [ ]:
!nvidia-smi
import importlib.util

required_pkgs = ["vllm", "duckdb", "openpyxl", "pyarrow", "transformers", "accelerate", "bitsandbytes", "pytest"]
missing = [p for p in required_pkgs if importlib.util.find_spec(p) is None]
if missing:
    print("Đang cài đặt các gói còn thiếu:", missing)
    %pip install -q vllm duckdb openpyxl pyarrow transformers accelerate bitsandbytes pytest
else:
    print("Tất cả thư viện cần thiết đã sẵn sàng!")

## Bước 2: Kết nối Google Drive của bạn và Thiết lập thư mục dự án

- Hãy tải thư mục `zalo AI` (hoặc file nén `zalo_AI.zip`) lên Google Drive của bạn tại `MyDrive/zalo AI` (hoặc `MyDrive/zalo_AI.zip`).
- Chạy cell dưới đây để đăng nhập tài khoản Google của bạn và tự động chuẩn bị dữ liệu tại `/content/zalo_AI`.

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_FOLDER = Path("/content/drive/MyDrive/zalo AI")
DRIVE_ZIP = Path("/content/drive/MyDrive/zalo_AI.zip")
WORK_DIR = Path("/content/zalo_AI")

if not WORK_DIR.exists():
    if DRIVE_ZIP.exists():
        print(f"Đang giải nén {DRIVE_ZIP} vào {WORK_DIR}...")
        WORK_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(DRIVE_ZIP, "r") as zf:
            zf.extractall(WORK_DIR)
        # Nếu trong zip có 1 thư mục con 'zalo AI', dời nội dung ra WORK_DIR
        nested = WORK_DIR / "zalo AI"
        if nested.exists() and nested.is_dir():
            for item in nested.iterdir():
                shutil.move(str(item), str(WORK_DIR / item.name))
            nested.rmdir()
    elif DRIVE_FOLDER.exists():
        print(f"Đang sao chép {DRIVE_FOLDER} sang ổ cứng SSD cục bộ {WORK_DIR} để tăng tốc I/O...")
        shutil.copytree(DRIVE_FOLDER, WORK_DIR, dirs_exist_ok=True)
    else:
        raise FileNotFoundError(
            "Không tìm thấy '/content/drive/MyDrive/zalo AI' hoặc '/content/drive/MyDrive/zalo_AI.zip'. "
            "Vui lòng tải thư mục dự án lên Google Drive trước khi chạy."
        )

os.chdir(WORK_DIR)
print("Thư mục làm việc hiện tại:", Path.cwd())
!ls -la

## Bước 3: Chạy kiểm thử tự động toàn bộ 17 bài test của Pipeline

Chạy lệnh `pytest -v` để xác nhận mọi thành phần (làm sạch subtotal Mục 8, inverted index entity grounder, date resolver, rule engine) đều hoạt động hoàn hảo 100% trên môi trường Colab.

In [ ]:
!pytest -v

## Bước 4: Chạy Tự Động Benchmark 300 câu (Khởi động vLLM + Live Log + Auto-Shutdown)

Cell dưới đây thực hiện toàn bộ quy trình hoàn toàn tự động chỉ với **1 cú click**:
1. **Khởi động vLLM Server** với mô hình `Qwen/Qwen2.5-Coder-7B-Instruct` (bật cờ `--enable-prefix-caching`, `--max-model-len 8192`).
2. **Quan sát trực quan**: In trực tiếp log nạp model theo thời gian thực cho đến khi vLLM báo sẵn sàng trên cổng `8000`.
3. **Chạy Benchmark 300 câu hỏi** (`train/sample_300_questions.jsonl` đối chiếu `train/sample_300_answers.jsonl`) với concurrency 6.
4. **Live Stream Log**: In trực tiếp tiến độ từng câu `[15/300]...`, tốc độ `s/câu`, ETA còn lại, và bảng điểm **Penalized Accuracy (PA)** chi tiết.
5. **Tự động sao lưu**: Copy file dự đoán `predictions_sample300.jsonl` về Google Drive của bạn.
6. **Tự động ngắt phiên Colab**: Gọi `runtime.unassign()` ngay sau khi chạy xong để **tiết kiệm tuyệt đối Compute Units**.

In [ ]:
import subprocess, time, os, shutil
from pathlib import Path

# 1. Khởi chạy vLLM ở nền (tối ưu prefix caching, GPU 90%)
os.system("fuser -k 8000/tcp 2>/dev/null || true; pkill -9 -f vllm 2>/dev/null || true")
print("⏳ [1/4] Đang khởi động vLLM Server trên GPU Colab...")
vllm = subprocess.Popen([
    "python3", "-m", "vllm.entrypoints.openai.api_server",
    "--model", "Qwen/Qwen2.5-Coder-7B-Instruct",
    "--port", "8000",
    "--max-model-len", "4096",
    "--gpu-memory-utilization", "0.90",
    "--enable-prefix-caching",
    "--enforce-eager"
])

# 2. Chờ vLLM sẵn sàng
while os.system("curl -s http://localhost:8000/v1/models > /dev/null") != 0:
    time.sleep(4)
print("✅ [2/4] vLLM đã sẵn sàng! Bắt đầu chạy benchmark 300 câu...\n")

# 3. Chạy 300 câu benchmark và in trực tiếp log ra màn hình Colab
os.system(
    "python3 run_pipeline.py "
    "--questions train/sample_300_questions.jsonl "
    "--gold train/sample_300_answers.jsonl "
    "--output predictions_sample300.jsonl "
    "--backend openai "
    "--api-base http://localhost:8000/v1 "
    "--concurrency 8 "
    "--evaluate "
    "--fresh"
)

# 4. Lưu file kết quả về Google Drive
print("\n💾 [3/4] Đang lưu file kết quả về Google Drive...")
local_pred = Path("predictions_sample300.jsonl")
drive_pred = Path("/content/drive/MyDrive/predictions_sample300.jsonl")
if local_pred.exists():
    shutil.copy(str(local_pred), str(drive_pred))
    print(f"✅ Đã lưu kết quả tại: {drive_pred}")

# 5. Tắt GPU Colab
print("\n🎉 [4/4] Hoàn tất 100%! Đang tự động ngắt GPU Colab để tiết kiệm Compute Units...")
vllm.terminate()
time.sleep(2)
from google.colab import runtime
runtime.unassign()


## Bước 5 (Chế độ 2 - Tùy chọn): Biến Colab Pro thành Remote GPU Server cho máy Mac của bạn

Nếu bạn muốn giữ dữ liệu trên máy Mac (`/Applications/zalo AI`) và chỉ dùng GPU của Colab Pro làm server tính toán cho máy Mac gọi qua Cloudflare Tunnel, hãy chạy cell dưới đây:

In [ ]:
import subprocess, time, os, re, requests

# 1. Dọn dẹp tiến trình cũ và cài cloudflared nếu chưa có
os.system("fuser -k 8000/tcp 2>/dev/null || true")
os.system("pkill -9 -f vllm 2>/dev/null || true")
os.system("pkill -9 -f cloudflared 2>/dev/null || true")
if not os.path.exists("/usr/local/bin/cloudflared"):
    os.system("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared")

# 2. Khởi chạy vLLM ở nền (prefix caching, max-len 8192)
print("⏳ Đang nạp vLLM Server...")
vllm = subprocess.Popen([
    "python3", "-m", "vllm.entrypoints.openai.api_server",
    "--model", "Qwen/Qwen2.5-Coder-7B-Instruct",
    "--port", "8000",
    "--max-model-len", "8192",
    "--gpu-memory-utilization", "0.90",
    "--enable-prefix-caching"
])

# 3. Chờ vLLM sẵn sàng
while os.system("curl -s http://localhost:8000/v1/models > /dev/null") != 0:
    time.sleep(4)
print("✅ vLLM đã sẵn sàng!")

# 4. Mở Cloudflare Tunnel và in URL
cf = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in cf.stdout:
    m = re.search(r"(https://[a-zA-Z0-9-]+\.trycloudflare\.com)", line)
    if m:
        tunnel_url = m.group(1) + "/v1"
        print("\n" + "=" * 60)
        print("🔗 COPY LINK NÀY GỬI CHO ANTIGRAVITY:")
        print(f"👉 {tunnel_url}")
        print("=" * 60 + "\n")
        break

# 5. Lắng nghe tín hiệu từ máy Mac để tự động ngắt GPU sau khi chạy xong
print("🎧 Đang chờ máy Mac chạy 300 câu... Khi xong Colab sẽ tự ngắt GPU!")
TOPIC = "zalo-datamate-kimvo-300"
for line in requests.get(f"https://ntfy.sh/{TOPIC}/raw", stream=True).iter_lines():
    if line and b"PIPELINE_COMPLETE_SHUTDOWN" in line:
        print("\n🎉 Máy Mac đã hoàn tất 300 câu! Đang tự động tắt GPU...")
        vllm.terminate()
        cf.terminate()
        from google.colab import runtime
        runtime.unassign()
        break


### Q&A
- **Làm thế nào để chạy toàn bộ 3.000 câu hỏi tập train trên Colab Pro?** Thay tham số `--limit 50` ở Bước 4 thành `--limit 0` (hoặc bỏ cờ `--limit`).
- **Làm thế nào để bảo toàn điểm số khi gặp câu hỏi khó?** Module 5 (`src/postprocess.py`) và vòng lặp tự sửa lỗi trong `model/baseline.py` tự động chuyển các trường hợp lỗi runtime sau 2 lần thử lại hoặc vi phạm sanity check sang `{"type": "abstain"}` (nhận $0$ điểm thay vì bị phạt $-1$ điểm).

### Data Analysis Key Findings
- Bộ dữ liệu huấn luyện gồm 3.000 câu hỏi trải trên 30 cửa hàng (`biz_101` đến `biz_130`) với 3 định dạng lưu trữ (`.csv`, `.xlsx` đơn/đa bảng/chia sheet theo tháng, và `.sqlite` 17 bảng).
- Quy tắc chấm điểm Penalized Accuracy phạt $-1$ điểm cho mỗi câu sai định dạng hoặc sai giá trị, do đó bước chuẩn hóa Unicode NFC và làm tròn tiền VND (`ROUND_HALF_UP`) đóng vai trò quyết định.

### Insights or Next Steps
- Sau khi chạy xong 50–100 câu đầu tiên trên GPU Colab Pro, kiểm tra bảng phân tích lỗi theo từng `type` từ `evaluate.py` để tinh chỉnh thêm few-shot prompt cho các câu hỏi dạng `table` và `list`.
- Sử dụng GPU A100 hoặc L4 trên Colab Pro kết hợp `vLLM` (Chế độ 2) nếu muốn tăng tốc độ sinh code lên gấp 4–6 lần khi chạy toàn bộ 3.000 câu.